In [ ]:
import os
import sys

import joblib
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, '../../..')
from utils.plotting import plot_movement_data
from utils.constants import Configs
from utils.loso_pipeline import run_loso_evaluation
from utils.helper_functions import load_outlier_data
from utils.loso_models.iforest_detector import IForestDetector
from utils.metrics import movement_level_confusion
from utils.bootstrap_utils import compute_bootstrap_ci_all_levels
from utils.helper_functions import save_loso_results
from utils.helper_functions import append_experiment_record
from utils.loso_pipeline import aggregate_to_trial_level, derive_trial_level_gt, summarize_level
from utils.feature_selection import make_corr_filter

TARGET_MOVEMENT = 'Park'
ALGORITHM_NAME = 'IForest'
MODEL_REPRESENTATION = 'FeatureBased'

N_BAYES_TRIALS = 40 
EXTRACT_SCORES = True

/home/hugo-guimar-es/Documents/GitHub/python_zeromq/Outlier_Removal/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/hugo-guimar-es/Documents/GitHub/python_zeromq/Outlier_Removal/.venv/lib/python3.13/site-packages/tslearn/bases/bases.py:16: UserWarning: h5py not installed, hdf5 features will not be supported.
Install h5py to use hdf5 features: http://docs.h5py.org/
  warn(h5py_msg)


In [2]:
# Load the processed data from the csv
cache_path = Configs.SMART_RETAIL_FILTERED_DATA_BIMANUAL
if not os.path.exists(cache_path):
    raise FileNotFoundError(f"Processed data not found at {cache_path}. Please run the data processing notebook first.")

df = pd.read_csv(cache_path)
df = df.sort_values(by=Configs.KEYS + ['step']).reset_index(drop=True)
df_filtered = df[df['movLabel'] == TARGET_MOVEMENT].copy()

In [3]:
# Metadata: one line per movement, same order as the matrices
results_temp_base = df_filtered[Configs.KEYS].drop_duplicates().reset_index(drop=True)
n_movements = len(results_temp_base)
print(f"Number of movements: {n_movements}")

Number of movements: 185


In [4]:
print("\n--- Fusing Outliers into a singular Ground Truth ---")

# Load the known outliers from the csv
outliers_left_csv, _  = load_outlier_data(Configs.SMART_RETAIL_OUTLIERS_LEFT_PATH)
outliers_right_csv, _ = load_outlier_data(Configs.SMART_RETAIL_OUTLIERS_RIGHT_PATH)

def prep_gt(gt, phase):
    """
    Prepare the ground truth data for a specific phase.
    """
    gt = gt[gt['movLabel'] == phase] if 'movLabel' in gt.columns else gt
    return gt.drop_duplicates(subset=Configs.KEYS).reset_index(drop=True)

gt_left  = prep_gt(outliers_left_csv,  TARGET_MOVEMENT)
gt_right = prep_gt(outliers_right_csv, TARGET_MOVEMENT)
gt_union = (pd.concat([gt_left, gt_right])
              .drop_duplicates(subset=Configs.KEYS).reset_index(drop=True))

GT = {'L': gt_left, 'R': gt_right, 'OR': gt_union}
for k, g in GT.items():
    print(f"GT {k}: {len(g)} positivos / {n_movements} movimentos = {len(g)/n_movements*100:.2f}%")

# All positive samples must exist in the metadata
base_keys = set(map(tuple, results_temp_base[Configs.KEYS].values))
for k, g in GT.items():
    missing = [t for t in map(tuple, g[Configs.KEYS].values) if t not in base_keys]
    assert not missing, f"GT {k}: {len(missing)} positivos sem movimento correspondente"


--- Fusing Outliers into a singular Ground Truth ---
Loading outlier data from /home/hugo-guimar-es/Documents/GitHub/python_zeromq/Data/SmartRetail/outliers/outliers_left.csv
Loading outlier data from /home/hugo-guimar-es/Documents/GitHub/python_zeromq/Data/SmartRetail/outliers/outliers_right.csv
GT L: 1 positivos / 185 movimentos = 0.54%
GT R: 3 positivos / 185 movimentos = 1.62%
GT OR: 4 positivos / 185 movimentos = 2.16%


### Load Features from csv

In [ ]:
cache_path = Configs.wrist_only_feature_extraction_features(TARGET_MOVEMENT)
if not os.path.exists(cache_path):
    raise FileNotFoundError(f"Features for {TARGET_MOVEMENT} not found at {cache_path}. Please run the feature extraction notebook first.")

features_df = joblib.load(cache_path)
features_df = features_df.sort_values(by=['sub', 'repID', 'movID']).reset_index(drop=True)
X_features = features_df[Configs.FEATURE_COLS].to_numpy()
n_movements_f = len(features_df)
total_trials = features_df.drop_duplicates(subset=['sub', 'repID']).shape[0]

assert not np.isnan(X_features).any(), "NaN in features"
assert len(features_df) == n_movements, (len(features_df), n_movements)
assert set(map(tuple, features_df[Configs.KEYS].values)) == base_keys
print(X_features.shape)

## Isolation Forest

In [6]:
# =========================================================================
# Base vars for future calculations 
# =========================================================================

n_positives = {k: len(g) for k, g in GT.items()}

# Ground truth used for evalution on this run
GT_KEY = 'OR'  # TODO isto vai ter de ficar noutro local, mas por agora é só para testar
gt_used = GT[GT_KEY]
total_known_outliers = len(gt_used)

# Movement-level anomalies
known_outlier_rate = total_known_outliers / n_movements if n_movements > 0 else 0.0
known_percentage = known_outlier_rate * 100

# print(f"✅ Sincronization OK. Lines: X={X_movement.shape[0]} | DTW={dtw_matrix.shape[0]} | metadata={n_movements}")
print(f"Phase: {TARGET_MOVEMENT}")
print(f"positives by ground truth: {n_positives}")
print(f"GT used ({"Union"}): {total_known_outliers} positives / {n_movements} movements = {known_percentage:.2f}%")

Phase: Park
positives by ground truth: {'L': 1, 'R': 3, 'OR': 4}
GT used (Union): 4 positives / 185 movements = 2.16%


### Grid Search

In [ ]:
FIXED_SEED = 42

# Cannot use itertools when I'm using Bayesian optimization
param_space = {
    'n_estimators': ('int', 50, 500),
    'max_samples': ('categorical', ['auto', 0.3, 0.5, 0.7, 0.9]),
    'contamination': ('categorical', Configs.CONTAMINATION_GRID),
    'random_state': FIXED_SEED
}

loso_results = run_loso_evaluation(
    detector_class=IForestDetector, 
    param_grid=param_space,
    data_matrix=X_features,                     # all features
    metadata_df=features_df[Configs.META_COLS], # same rows as X_features
    known_outliers_df=gt_used,
    search_type='bayesian',
    n_bayes_trials=N_BAYES_TRIALS,
    metric_type='euclidean',
    scale_features=True,
    compute_trial_level=False,
    extract_scores=EXTRACT_SCORES,
    print_results=False,
    feature_filter=make_corr_filter(Configs.FEATURE_COLS, Configs.FEATURES_CORRELATION_THRESHOLD, Configs.PREFER_DROP),
)

LOSO Cross-Validation with 19 subjects (4 known positive movements)...
Evaluating Subject: 1 | Evaluating Subject: 2 | Evaluating Subject: 3 | Evaluating Subject: 4 | Evaluating Subject: 5 | Evaluating Subject: 6 | Evaluating Subject: 7 | Evaluating Subject: 8 | Evaluating Subject: 9 | Evaluating Subject: 10 | Evaluating Subject: 11 | Evaluating Subject: 12 | Evaluating Subject: 13 | Evaluating Subject: 14 | Evaluating Subject: 15 | Evaluating Subject: 16 | Evaluating Subject: 17 | Evaluating Subject: 18 | Evaluating Subject: 19 | 

In [8]:
# n_estimators: Número de árvores na floresta. Mais árvores podem melhorar a detecção, mas aumentam o tempo de treino.
# contamination: Auto é uma boa opção para começar, pois estima a proporção de outliers com base nos dados.
# random_state was set with the answer to the life, universe and everything (for reproducibility)
# max_samples: 'auto' $$max\_samples = \min(256, n\_samples)$$

best_params = loso_results['Best_Params']
n_estimators = best_params['n_estimators']
contamination = best_params['contamination']
max_samples = best_params['max_samples']

final_scaler = StandardScaler()
X_features_scaled_full = final_scaler.fit_transform(X_features)

##  Report Seed Variance

In [9]:
# Extract the exact hyperparameter dictionary used for each specific subject in the original run
fold_params = {
    fold['subject_out']: fold['params'].copy()
    for fold in loso_results['Fold_History']
}

In [ ]:
# --- Variance Evaluation for Stochastic Model ---
test_seeds = [10, 42, 123, 999, 2026]
f2_scores_seeds = []
precision_scores = []
recall_scores = []
auprc_scores = []
mcc_scores = []
all_seed_results = []

for seed in test_seeds:
    # Single configuration grid with changing seed
    param_space_seed = {
        'n_estimators': ('int', 50, 500),
        'max_samples': ('categorical', ['auto', 0.3, 0.5, 0.7, 0.9]),
        'contamination': ('categorical', Configs.CONTAMINATION_GRID),
        'random_state': seed
    }

    seed_results = run_loso_evaluation(
        detector_class=IForestDetector, 
        param_grid=param_space_seed,
        data_matrix=X_features,                     # all features
        metadata_df=features_df[Configs.META_COLS], # same rows as X_features
        known_outliers_df=gt_used,
        search_type='bayesian',
        n_bayes_trials=N_BAYES_TRIALS,
        metric_type='euclidean',
        fixed_fold_params=fold_params,
        override_random_state=seed, 
        scale_features=True,
        compute_trial_level=False,
        extract_scores=EXTRACT_SCORES,
        print_results=False,
        feature_filter=make_corr_filter(Configs.FEATURE_COLS, Configs.FEATURES_CORRELATION_THRESHOLD, Configs.PREFER_DROP),
    )

    all_seed_results.append(seed_results) 
    f2_scores_seeds.append(seed_results['Movement']['F2-Score'])
    precision_scores.append(seed_results['Movement']['Precision'])
    recall_scores.append(seed_results['Movement']['Recall'])
    mcc_scores.append(seed_results['Movement']['MCC'])
    
    if EXTRACT_SCORES:
        auprc_scores.append(seed_results['Movement']['AUPRC'])
    
print("\n\n-- IForest Stability Results (5 seeds) ---")
print(f"F2-Score:  {np.mean(f2_scores_seeds):.2f}% ± {np.std(f2_scores_seeds):.2f}%")
print(f"Precision: {np.mean(precision_scores):.2f}% ± {np.std(precision_scores):.2f}%")
print(f"Recall:    {np.mean(recall_scores):.2f}% ± {np.std(recall_scores):.2f}%")
print(f"MCC:       {np.mean(mcc_scores):.2f}% ± {np.std(mcc_scores):.2f}%")
print(f"AUPRC:     {np.mean(auprc_scores):.2f}% ± {np.std(auprc_scores):.2f}%")
print("-------------------------------------------")

LOSO Cross-Validation with 19 subjects (4 known positive movements)...
Evaluating Subject: 1 | Evaluating Subject: 2 | Evaluating Subject: 3 | Evaluating Subject: 4 | Evaluating Subject: 5 | Evaluating Subject: 6 | Evaluating Subject: 7 | Evaluating Subject: 8 | Evaluating Subject: 9 | Evaluating Subject: 10 | Evaluating Subject: 11 | Evaluating Subject: 12 | Evaluating Subject: 13 | Evaluating Subject: 14 | Evaluating Subject: 15 | Evaluating Subject: 16 | Evaluating Subject: 17 | Evaluating Subject: 18 | Evaluating Subject: 19 | LOSO Cross-Validation with 19 subjects (4 known positive movements)...
Evaluating Subject: 1 | Evaluating Subject: 2 | Evaluating Subject: 3 | Evaluating Subject: 4 | Evaluating Subject: 5 | Evaluating Subject: 6 | Evaluating Subject: 7 | Evaluating Subject: 8 | Evaluating Subject: 9 | Evaluating Subject: 10 | Evaluating Subject: 11 | Evaluating Subject: 12 | Evaluating Subject: 13 | Evaluating Subject: 14 | Evaluating Subject: 15 | Evaluating Subject: 16 | E

In [11]:
oof_movement_per_seed = [sr['OOF_Movement'] for sr in all_seed_results]

reference_keys = oof_movement_per_seed[0][['sub', 'repID', 'movID', 'true_is_outlier']].reset_index(drop=True)
for oof in oof_movement_per_seed[1:]:
    keys = oof[['sub', 'repID', 'movID', 'true_is_outlier']].reset_index(drop=True)
    assert reference_keys.equals(keys), "As linhas de OOF_Movement divergem entre seeds."

stacked_preds = np.stack([oof['pred_is_outlier'].values for oof in oof_movement_per_seed], axis=0)
consensus_pred = (stacked_preds.mean(axis=0) >= 0.5).astype(int)

if all(oof['anomaly_score'].notna().all() for oof in oof_movement_per_seed):
    stacked_scores = np.stack([oof['anomaly_score'].values for oof in oof_movement_per_seed], axis=0)
    consensus_score = stacked_scores.mean(axis=0)
else:
    consensus_score = np.full(len(reference_keys), np.nan)

oof_movement_consensus = reference_keys.copy()
oof_movement_consensus['pred_is_outlier'] = consensus_pred
oof_movement_consensus['anomaly_score'] = consensus_score

trial_outliers_gt = derive_trial_level_gt(gt_used)
oof_trial_consensus = aggregate_to_trial_level(oof_movement_consensus, trial_outliers_gt, score_col='anomaly_score')

movement_summary_consensus = summarize_level(oof_movement_consensus, 'movement', len(gt_used))
trial_summary_consensus = summarize_level(oof_trial_consensus, 'trial', len(trial_outliers_gt))

In [12]:
loso_results_consensus = dict(loso_results)
loso_results_consensus['Movement'] = movement_summary_consensus
loso_results_consensus['OOF_Movement'] = oof_movement_consensus
loso_results_consensus['Blind_Y_True'] = oof_movement_consensus['true_is_outlier'].values
loso_results_consensus['Blind_Y_Pred'] = oof_movement_consensus['pred_is_outlier'].values

print("--- IForest (Feature-Based, Seed-Robust Consensus) -- nível de MOVIMENTO ---")
for k in ['Precision', 'Recall', 'F1-Score', 'F2-Score', 'MCC', 'AUPRC', 'AUROC', 'Rejection_Rate']:
    v = movement_summary_consensus.get(k, np.nan)
    print(f"  {k}: {v:.4f}" if isinstance(v, float) and not np.isnan(v) else f"  {k}: {v}")

--- IForest (Feature-Based, Seed-Robust Consensus) -- nível de MOVIMENTO ---
  Precision: 0.1176
  Recall: 0.5000
  F1-Score: 0.1905
  F2-Score: 0.3030
  MCC: 0.2100
  AUPRC: 0.4291
  AUROC: 0.9572
  Rejection_Rate: 9.1892


## Bootstrap CI interval 95%

In [13]:
ci_results_consensus = compute_bootstrap_ci_all_levels(loso_results_consensus, compute_scores=EXTRACT_SCORES, seed=42)

In [15]:
save_loso_results(
    model_name=ALGORITHM_NAME,
    representation=MODEL_REPRESENTATION,
    target_movement=TARGET_MOVEMENT,
    loso_results=loso_results_consensus,
)

append_experiment_record(
    summary_path=Configs.loso_model_summary_path(MODEL_REPRESENTATION, GT_KEY),
    phase=TARGET_MOVEMENT,
    representation=MODEL_REPRESENTATION,
    algorithm=ALGORITHM_NAME,
    total_samples=n_movements,
    loso_results=loso_results_consensus,
    bootstrap_results=ci_results_consensus,
)


✅ Experiment Summary for IForest - FeatureBased - Park - 185 Movements (MOVEMENT-level):
Unique Outliers Detected: 17
Rejection Rate: 9.19%
Known Positives in Ground Truth: 4
------------------------------
Total True Positives (Global):  2
Total False Positives (Global): 15
Total False Negatives (Global): 2
Total True Negatives (Global):  166
------------------------------
Blind Precision: 11.76 [0.00–27.13]
Blind Recall:    50.00 [0.00–100.00]
Blind F1-Score:  19.05
Blind F2-Score:  30.30 [0.00–57.14]
Blind MCC:       0.21 [-0.04–0.45]
Blind AUPRC:  42.91 [10.00–100.00]
Blind AUROC:   95.72

Total Tuning Time (19 Subjects): 588.8421s
Total Inference Time (19 Subjects): 7.7488s

Overall Best Parameters (Most frequent across folds):
   {'contamination': 0.12, 'max_samples': 'auto', 'n_estimators': 117, 'random_state': 42}
Occurrences: 2


### Train Model
Aqui não existe um treino de um modelo final, mas sim a agregação do voto maioritário do LOSO + teste de Seeds.

In [16]:
consensus_outlier_keys = set(
    zip(oof_trial_consensus[oof_trial_consensus['pred_is_outlier'] == 1]['sub'],
        oof_trial_consensus[oof_trial_consensus['pred_is_outlier'] == 1]['repID'])
)
print(f"✅ {len(consensus_outlier_keys)} trials marcados como outlier pelo consenso.")

✅ 16 trials marcados como outlier pelo consenso.


In [ ]:
features_df['anomaly'] = features_df.apply(
    lambda row: -1 if (row['sub'], row['repID']) in consensus_outlier_keys else 1,
    axis=1
) # -1 for outlier, 1 for normal

n_outliers =  len(consensus_outlier_keys)
total_trials_consensus = len(oof_trial_consensus)
correct_trash_pct = (n_outliers / total_trials_consensus) * 100 if total_trials_consensus > 0 else 0.0
n_unique_outliers = n_outliers

print(f"Unique Outlier Trials Detected (Consensus): {n_unique_outliers}")
print(f"Rejection Rate (Consensus): {correct_trash_pct:.2f}%")

outliers_details = features_df[features_df['anomaly'] == -1][['sub', 'repID', 'movID']]

Unique Outlier Trials Detected (Consensus): 16
Rejection Rate (Consensus): 16.84%


In [18]:
features_df['pred_is_outlier'] = (features_df['anomaly'] == -1).astype(int)
trial_level_df = features_df.groupby(['sub', 'repID'])['pred_is_outlier'].max().reset_index()

gt_keys = set(zip(gt_used['sub'], gt_used['repID']))

trial_level_df['true_is_outlier'] = trial_level_df.apply(
    lambda row: 1 if (row['sub'], row['repID']) in gt_keys else 0,
    axis=1
)

y_true_final = trial_level_df['true_is_outlier'].values
y_pred_final = trial_level_df['pred_is_outlier'].values

In [19]:
cols_to_extract = ['sub', 'repID', 'movID']
outliers_metadata_df = features_df.loc[outliers_details.index, cols_to_extract].copy()
print("Quem são os outliers reais?")
print(outliers_metadata_df)

Quem são os outliers reais?
     sub  repID  movID
1      1      2      9
8      2      4      9
9      2      5      9
20     5      1      9
23     5      4      9
44     9      5      9
45    10      1      9
46    10      2      9
52    11      3      9
80    14      1      9
89    15      2      9
90    15      2     10
91    15      2     11
92    15      2     12
141   17      5      9
142   17      5     10
143   17      5     11
144   17      5     12
145   18      1      9
146   18      1     10
147   18      1     11
148   18      1     12
149   18      2      9
150   18      2     10
151   18      2     11
152   18      2     12
157   18      4      9
158   18      4     10
159   18      4     11
160   18      4     12
161   18      5      9
162   18      5     10
163   18      5     11
164   18      5     12


In [ ]:
identified_unique = outliers_metadata_df.drop_duplicates(subset=['sub', 'repID'])

matched_outliers = pd.merge(identified_unique, gt_used, on=['sub', 'repID'], how='inner')

missing_outliers = pd.merge(gt_used, identified_unique, on=['sub', 'repID'], how='left', indicator=True)
missing_outliers = missing_outliers[missing_outliers['_merge'] == 'left_only']

metrics = movement_level_confusion(features_df, np.where(features_df['anomaly'].values == -1)[0], gt_used, Configs.KEYS)

--- Confusion Matrix (Based on sub+repID) ---
True Positives (Identified and in CSV) : 2
False Positives (Identified, not in CSV): 14
False Negatives (In CSV, not identified): 2
True Negatives (Normal, stayed normal): 77
--------------------------------------------------
Missing outliers (In CSV, not identified):
   sub  repID
0   16      2
1    1      3

Precision (False Positives): 2/16 (12.50%)
Recall (True Positive Rate): 2/4 (50.00%)
F1-Score: 20.00%
F2-Score: 31.25%


In [ ]:
outlier_ids = features_df[features_df['anomaly'] == -1][['sub', 'repID', 'movID']]

# Create a "unique key" to simplify filtering
features_df['key'] = features_df['sub'].astype(str) + '_' + features_df['repID'].astype(str) + '_' + features_df['movID'].astype(str)
outlier_keys = features_df[features_df['anomaly'] == -1]['key'].tolist()

# Return to the original DataFrame for plotting
# Create a temporary key in the original df as well
df['temp_key'] = df['sub'].astype(str) + '_' + df['repID'].astype(str) + '_' + df['movID'].astype(str)

# Separate visually
# Note: This is only for plotting; it does not permanently alter the original df if you do not want it to
df_outliers = df[df['temp_key'].isin(outlier_keys) & (df['movLabel'] == TARGET_MOVEMENT)]
df_clean = df[~df['temp_key'].isin(outlier_keys) & (df['movLabel'] == TARGET_MOVEMENT)]

plot_movement_data(
    clean_data=df_clean,
    outlier_data=df_outliers,
    target_movement=TARGET_MOVEMENT,
    algorithm_name=ALGORITHM_NAME,
    save_fig=Configs.SAVE_PLOTS,
    model_representation=MODEL_REPRESENTATION,
    save_path=Configs.DESKTOP_PATH
)

SyntaxError: keyword argument repeated: model_representation (1236292763.py, line 24)

In [ ]:
df.drop(columns=['temp_key'], inplace=True)